# 08 — Validate real JINA/libnucnet XML files

This notebook uses your real JINA-style files: `nuclides.xml`, `reaction_data.xml`, `zone.xml`, and an optional output XML from the original workflow.

It checks whether the pure-Python reader can parse the database, preserve the initial mass fractions, identify missing species, validate A/Z conservation, and evaluate sample reaction rates.

In [ ]:
from pathlib import Path
import sys, json

# If running from the source tree without installation:
sys.path.insert(0, str(Path.cwd().parent / "src"))

from nucnetpy.io.jina import read_jina_xml
from nucnetpy.io.xml import read_xml

## File paths

Place your XML files in `notebooks/data/`, or edit the paths below.

In [ ]:
DATA = Path("data")
NUCLIDES_XML = DATA / "nuclides.xml"
REACTIONS_XML = DATA / "reaction_data.xml"
ZONE_XML = DATA / "zone.xml"
OUTPUT_XML = DATA / "output_Nova_exp.xml"

for p in [NUCLIDES_XML, REACTIONS_XML, ZONE_XML, OUTPUT_XML]:
    print(p, "exists=", p.exists())

## Load the JINA nuclide/reaction database and initial zone

In [ ]:
net = read_jina_xml(NUCLIDES_XML, REACTIONS_XML, ZONE_XML)
validation = net.validate()

print("species:", len(net.species))
print("reactions:", len(net.reactions.reactions))
print("zones:", len(net.zones))
print("missing species:", len(validation["missing_species"]))
print("invalid reactions:", len(validation["invalid_reactions"]))

## Check the initial zone

The zone file stores mass fractions `X`; NucNetPy converts them to abundances `Y = X/A`. The mass-fraction sum should remain close to 1.

In [ ]:
zone = net.zones[0]
print("number of abundances:", len(zone.abundances))
print("sum Y:", zone.ysum())
print("sum X:", zone.xsum(net.species))
print("Ye:", zone.ye(net.species))
print("T9:", zone.temperature9())
print("rho:", zone.density())
print("optional properties:")
for k, v in zone.optional_properties.items():
    print(f"  {k}: {v}")

## Evaluate sample reaction rates

In [ ]:
T9 = zone.temperature9()
rho = zone.density()

for r in net.reactions.reactions[:10]:
    print(f"{r.string:70s}  rate={r.rate(T9, rho):.6e}  fits={len(r.rate_fits)}  constant={r.constant_rate}")

## Load and inspect the original output XML

In [ ]:
out = read_xml(OUTPUT_XML)
print("species:", len(out.species))
print("reactions:", len(out.reactions.reactions))
print("zones:", len(out.zones))

z0 = out.zones[0]
zf = out.zones[-1]
print("zone 0: Xsum=", z0.xsum(out.species), "T9=", z0.temperature9(), "rho=", z0.density())
print("final zone: Xsum=", zf.xsum(out.species), "T9=", zf.temperature9(), "rho=", zf.density())

## Export a JSON validation summary

In [ ]:
summary = {
    "input_database": {
        "species": len(net.species),
        "reactions": len(net.reactions.reactions),
        "zones": len(net.zones),
        "missing_species": len(validation["missing_species"]),
        "invalid_reactions": len(validation["invalid_reactions"]),
    },
    "initial_zone": {
        "abundance_count": len(zone.abundances),
        "xsum": zone.xsum(net.species),
        "ysum": zone.ysum(),
        "Ye": zone.ye(net.species),
        "T9": zone.temperature9(),
        "rho": zone.density(),
    },
    "output_xml": {
        "species": len(out.species),
        "reactions": len(out.reactions.reactions),
        "zones": len(out.zones),
        "zone0_xsum": z0.xsum(out.species),
        "final_zone_xsum": zf.xsum(out.species),
    }
}
Path("real_xml_validation_summary.json").write_text(json.dumps(summary, indent=2))
summary